# 8. Cause Normalization

This notebook normalizes the raw mechanical cause text from the `Caused of Problem` column into standardized, high-level `cause_canonical` (RootCausePattern) categories.

**Approach:**
1. **Embeddings:** Vectorize all unique cause texts using the SentenceTransformer model.
2. **Dimensionality Reduction:** Project embeddings to a lower-dimensional space using UMAP.
3. **Clustering:** Identify dense, semantic groups of causes using HDBSCAN.
4. **LLM-assisted Labeling:** Prompt the local LLM (`qwen2.5:3b`) to summarize representative samples from each cluster into a concise (3-4 words) canonical label.
5. **Save Enriched Data:** Map the labels back to all EMR rows and save the enriched dataset to `output/clustered_emr.csv`.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import umap
import hdbscan
from sentence_transformers import SentenceTransformer
from langchain_core.messages import SystemMessage, HumanMessage

# Ultimate foolproof path injection to find 'src' regardless of current working directory
cwd = os.path.abspath(os.getcwd())
project_root = None

# 1. Walk up from current working directory
temp = cwd
for _ in range(4):
    if os.path.exists(os.path.join(temp, 'src', 'config.py')):
        project_root = temp
        break
    parent = os.path.abspath(os.path.join(temp, '..'))
    if parent == temp:
        break
    temp = parent

if project_root:
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
    path_prefix = os.path.relpath(project_root, cwd)
    print(f"Project root found: {project_root}")
else:
    print("Warning: Project root not automatically detected. Using defaults.")
    path_prefix = ".."

from src.config import settings
from src.utils import get_llm

d:\PEMROGRAMAN\LLM-DESKTOP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root found: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator


In [2]:
print("1. Loading clustered EMR data...")
file_path = os.path.join(path_prefix, "output", "clustered_emr.csv")
df = pd.read_csv(file_path)
print(f"Loaded {len(df)} records.")

cause_col = "Caused of Problem"
raw_causes = df[cause_col].fillna("").astype(str).str.strip()
valid_mask = raw_causes.str.len() > 3

# Get unique cause texts to process efficiently
unique_causes = raw_causes[valid_mask].unique()
print(f"Total unique cause texts to process: {len(unique_causes)}")

1. Loading clustered EMR data...
Loaded 20630 records.
Total unique cause texts to process: 16488


In [3]:
print("2. Generating embeddings using SentenceTransformer...")
transformer_model = SentenceTransformer(settings.embedding_model)
embeddings = transformer_model.encode(
    unique_causes.tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)
print(f"Embeddings shape: {embeddings.shape}")

2. Generating embeddings using SentenceTransformer...


Batches: 100%|██████████| 258/258 [03:10<00:00,  1.35it/s]

Embeddings shape: (16488, 384)


In [4]:
print("3. Running UMAP dimensionality reduction...")
reducer = umap.UMAP(
    n_components=10,
    n_neighbors=25,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)
reduced_embeddings = reducer.fit_transform(embeddings)

print("4. Running HDBSCAN clustering...")
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=settings.hdbscan_min_cluster_size,
    min_samples=settings.hdbscan_min_samples,
    metric="euclidean",
    cluster_selection_method="eom"
)
labels = clusterer.fit_predict(reduced_embeddings)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = int((labels == -1).sum())
print(f"Found {n_clusters} clusters, {n_noise} noise points ({n_noise/len(labels)*100:.1f}%)")

mapping_df = pd.DataFrame({
    "raw_text": unique_causes,
    "cluster_id": labels
})

d:\PEMROGRAMAN\LLM-DESKTOP\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


3. Running UMAP dimensionality reduction...
4. Running HDBSCAN clustering...
Found 715 clusters, 5510 noise points (33.4%)


In [5]:
print("5. Labeling clusters with local LLM...")
llm = get_llm(temperature=0.0)
cluster_labels = {-1: "Penyebab Lain / Khusus"}

for c_id in sorted(list(set(labels))):
    if c_id == -1:
        continue
        
    # Get representative sample texts from this cluster
    samples = mapping_df[mapping_df["cluster_id"] == c_id]["raw_text"].head(8).tolist()
    examples_str = "\n".join([f"- {s[:150]}" for s in samples])
    
    prompt = f"""Anda adalah senior reliability engineer alat berat yang bertugas merumuskan akar masalah (Root Cause) secara sangat spesifik.
Diberikan beberapa contoh deskripsi penyebab kerusakan di lapangan dari mekanik:
{examples_str}

Tugas Anda:
Tentukan 1 label akar masalah (Root Cause) berupa FRASE yang spesifik, baku, dan jelas menggambarkan komponen serta jenis kegagalan fisiknya (maksimal 3-5 kata dalam Bahasa Indonesia formal).

ATURAN PEMBUATAN LABEL:
1. Label HARUS spesifik dan mendetail menyebutkan komponen apa yang bermasalah dan bagaimana kondisinya (Contoh gaya penulisan yang benar: "Kebocoran Seal Final Drive LH", "Kelonggaran Bolt Mounting", "Keretakan Track Frame", "Penyumbatan Filter Transmisi").
2. JANGAN gunakan kata-kata umum seperti "Penyebab Lain", "Kerusakan Komponen", "Mechanical Error", "Aus Umur Pakai", "Korsleting Kabel" secara umum saja. Spesifikkan komponen mana yang aus/korsleting (misal: "Kausan Hub Roda" atau "Korsleting Kabel Sensor Temp").
3. JANGAN meniru contoh gaya penulisan di atas secara mentah-mentah jika isi contoh teks di atas membahas topik komponen yang berbeda.
4. Output HANYA berupa teks label tersebut saja tanpa tanda petik, penjelasan, bullet point, atau kata pengantar apapun."""
    
    try:
        messages = [
            SystemMessage(content="You are a precise data labeling assistant. Output only the requested label."),
            HumanMessage(content=prompt)
        ]
        response = llm.invoke(messages)
        label = response.content.strip().replace('"', '').replace("'", "")
        if label.endswith("."):
            label = label[:-1]
        print(f"  Cluster {c_id} (count: {len(mapping_df[mapping_df['cluster_id'] == c_id])}) -> {label}")
        cluster_labels[c_id] = label
    except Exception as e:
        fallback = f"Penyebab Klaster {c_id}"
        print(f"  Cluster {c_id} -> FAILED ({e}), using fallback: {fallback}")
        cluster_labels[c_id] = fallback

5. Labeling clusters with local LLM...
  Cluster 0 (count: 11) -> Penyumbatan Filter Sekunder Udara
  Cluster 1 (count: 45) -> Kebocoran Valve Core RH Front Suspensi
  Cluster 2 (count: 8) -> Kebocoran Hose EGR
  Cluster 3 (count: 21) -> Kebocoran Bushing Antiroll Bar
  Cluster 4 (count: 16) -> Kebocoran Seal FC Pump
  Cluster 5 (count: 5) -> Kebocoran Seal Floating Assy
  Cluster 6 (count: 65) -> Kondisi Abnormal Damper Engine
  Cluster 7 (count: 6) -> Kebocoran Seal Final Drive LH
  Cluster 8 (count: 8) -> Kebocoran Seal Equalizer Bar
  Cluster 9 (count: 5) -> Kebocoran EGR Cooler
  Cluster 10 (count: 13) -> Installasi Block Plate EGR Cooler Salah
  Cluster 11 (count: 11) -> Kebocoran Fan Motor Radiator Internal
  Cluster 12 (count: 48) -> Oblak Track Roller LH No 3, 5, 6
  Cluster 13 (count: 7) -> Kegagalan Horn Elektrikal
  Cluster 14 (count: 8) -> Kebocoran Oring Penyekat
  Cluster 15 (count: 7) -> Kebocoran Seal ADT743
  Cluster 16 (count: 8) -> Kebocoran Oli Cylinder Steer
  Clu

In [6]:
print("6. Mapping cause_canonical and saving CSV...")
text_to_canonical = dict(zip(mapping_df["raw_text"], mapping_df["cluster_id"].map(cluster_labels)))
df["cause_canonical"] = raw_causes.map(text_to_canonical).fillna("Penyebab Tidak Terdefinisi")

# Save the updated CSV
output_file = os.path.join(path_prefix, "output", "clustered_emr.csv")
df.to_csv(output_file, index=False)
print(f"Successfully saved updated clustered EMR data to: {output_file}")

# Preview sample mappings
df[[cause_col, "cause_canonical"]].drop_duplicates().head(10)

6. Mapping cause_canonical and saving CSV...
Successfully saved updated clustered EMR data to: ..\output\clustered_emr.csv


,Caused of Problem,cause_canonical
0,Dari hasil analisa pengecekan berkurangnya fin...,Kebocoran Seal Final Drive LH
1,DATA 1. HM 3738 2. Informasi dari operator ket...,Kebocoran Pressure Adjuster
2,Penyebab Syteering cylinder leaks dikarenakan ...,Kebocoran Seal Blade Tilt Cylinder
3,Bolt stooper cabin tidak terpasang,Kelonggaran Bolt Mounting Bottom Guard Transmisi
4,Dilakukan pemeriksaan oleh mekanik dan ditemuk...,Penyebab Lain / Khusus
5,DATA: * Unit bergetar ketika travel. * Kondisi...,Penyebab Lain / Khusus
6,DATA: * Unit bergetar ketika travel. * Kondisi...,Penyebab Lain / Khusus
7,Dilaporkan terjadi problem Bolt mounting Cap T...,Penyebab Lain / Khusus
8,MOUNTING FUEL TANK BROKEN (SISI BELAKANG BAWAH...,Keretakan Fuel Tank LH
9,BOLT MOUNTING FUEL TANK BROKEN TIDAK ADA TANDA...,Keretakan Fuel Tank LH


In [ ]:
print("7. Recovering Uncategorized Data via Contextual Labeling...")
from langchain_core.messages import HumanMessage
import numpy as np

# 1. Identify valid cluster centroids
valid_cids = [c for c in cluster_labels.keys() if c != -1]
centroids = {}
for cid in valid_cids:
    mask = labels == cid
    c_emb = embeddings[mask]
    centroids[cid] = c_emb.mean(axis=0)

# 2. Process noise points
noise_mask = labels == -1
noise_indices = np.where(noise_mask)[0]

recovered_count = 0
auto_count = 0
llm_count = 0

for i, idx in enumerate(noise_indices):
    raw_text = unique_causes[idx]
    n_emb = embeddings[idx]
    
    sims = [(cid, np.dot(n_emb, cent)/(np.linalg.norm(n_emb)*np.linalg.norm(cent))) for cid, cent in centroids.items()]
    sims.sort(key=lambda x: x[1], reverse=True)
    if not sims: continue
    
    top_5 = sims[:5]
    best_cid, best_sim = top_5[0]
    matched_cid = -1
    
    if best_sim >= 0.82:
        matched_cid = best_cid
        auto_count += 1
    else:
        options_text = "\n".join([f"{j+1}. {cluster_labels[cid]}" for j, (cid, sim) in enumerate(top_5)])
        prompt = f"""Anda adalah pakar maintenance alat berat.
Teks asli laporan mekanik: "{raw_text}"

Teks ini sebelumnya tidak terklasifikasi. Berdasarkan konteks kerusakan, pilih SATU label yang paling relevan dan akurat dari 5 daftar kategori standar berikut:
{options_text}

PENTING: Jika teks asli tersebut murni karakter acak (typo parah), tidak masuk akal, atau sama sekali tidak berhubungan dengan 5 opsi di atas, Anda WAJIB menjawab: "Uncategorized".
Output HANYA teks label pilihan Anda tanpa tambahan apapun."""
        try:
            response = llm.invoke([HumanMessage(content=prompt)])
            ans = response.content.strip().replace('"', '').replace("'", "")
            
            valid_options = [cluster_labels[cid] for cid, sim in top_5]
            matched_label = "Uncategorized"
            for opt in valid_options:
                if opt.lower() in ans.lower():
                    matched_label = opt
                    break
                    
            if matched_label != "Uncategorized":
                matched_cid = [cid for cid, sim in top_5 if cluster_labels[cid] == matched_label][0]
                llm_count += 1
        except Exception:
            pass

    if matched_cid != -1:
        labels[idx] = matched_cid
        mapping_df.loc[idx, "cluster_id"] = matched_cid
        recovered_count += 1

    if (i + 1) % 100 == 0:
        print(f"  Processed {i+1}/{len(noise_indices)} noise records...")

print(f"\nSuccessfully recovered {recovered_count} noise records (Auto: {auto_count}, LLM: {llm_count}).")

# 3. Remap and Save
print("8. Re-saving enriched dataset...")
text_to_canonical = dict(zip(mapping_df["raw_text"], mapping_df["cluster_id"].map(cluster_labels).fillna("Penyebab Lain / Khusus")))
df["cause_canonical"] = raw_causes.map(text_to_canonical).fillna("Penyebab Tidak Terdefinisi")

output_file = os.path.join(path_prefix, "output", "clustered_emr_with_causes.csv")
df.to_csv(output_file, index=False)
print("Done.")

7. Recovering Uncategorized Data via Contextual Labeling...
  Processed 100/5510 noise records...
  Processed 200/5510 noise records...
  Processed 300/5510 noise records...
  Processed 400/5510 noise records...
  Processed 500/5510 noise records...
  Processed 600/5510 noise records...
  Processed 700/5510 noise records...
  Processed 800/5510 noise records...
  Processed 900/5510 noise records...
  Processed 1000/5510 noise records...
  Processed 1100/5510 noise records...
  Processed 1200/5510 noise records...
  Processed 1300/5510 noise records...
  Processed 1400/5510 noise records...
  Processed 1500/5510 noise records...
  Processed 1600/5510 noise records...
  Processed 1700/5510 noise records...
  Processed 1800/5510 noise records...
  Processed 1900/5510 noise records...
  Processed 2000/5510 noise records...
  Processed 2100/5510 noise records...
  Processed 2200/5510 noise records...
  Processed 2300/5510 noise records...
  Processed 2400/5510 noise records...
  Processed 2

: 